In [ ]:
!pip install PySastrawi==1.2.0 nltk==3.9.1 -q

In [ ]:
import pandas as pd
import numpy as np
import re
import string
import os
from sklearn.model_selection import StratifiedKFold
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
import nltk
nltk.download('stopwords')
nltk.download('punkt')
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

In [ ]:
factory = StemmerFactory()
stemmer_id = factory.create_stemmer()

stopword_factory = StopWordRemoverFactory()
stopwords_id = set(stopword_factory.get_stop_words())

stopwords_en = set(stopwords.words('english'))
stemmer_en = PorterStemmer()

print(f"Stopwords ID: {len(stopwords_id)}")
print(f"Stopwords EN: {len(stopwords_en)}")

In [ ]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
def preprocess_id(text, level):
    """Bahasa Indonesia"""
    if not isinstance(text, str):
        text = str(text)
    if level == 0:
        return text

    # Level 1: Case folding
    text = text.lower()
    if level == 1:
        return text

    # Level 2: Cleaning
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'\S+@\S+', ' ', text)
    text = re.sub(r'@\w+|#\w+', ' ', text)
    text = re.sub(r'\d+', ' ', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'[^\x00-\x7f]', ' ', text)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    text = re.sub(r'\s+', ' ', text).strip()
    if level == 2:
        return text

    # Level 3: Stopword removal ID
    tokens = text.split()
    tokens = [t for t in tokens if t not in stopwords_id and len(t) > 1]
    text = ' '.join(tokens)
    if level == 3:
        return text

    # Level 4: Stemming Sastrawi
    text = stemmer_id.stem(text)
    return text


def preprocess_en(text, level):
    """English"""
    if not isinstance(text, str):
        text = str(text)
    if level == 0:
        return text

    # Level 1: Case folding
    text = text.lower()
    if level == 1:
        return text

    # Level 2: Cleaning
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'\S+@\S+', ' ', text)
    text = re.sub(r'\d+', ' ', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip()
    if level == 2:
        return text

    # Level 3: Stopword removal EN
    tokens = text.split()
    tokens = [t for t in tokens if t not in stopwords_en and len(t) > 1]
    text = ' '.join(tokens)
    if level == 3:
        return text

    # Level 4: Stemming Porter
    tokens = text.split()
    tokens = [stemmer_en.stem(t) for t in tokens]
    text = ' '.join(tokens)
    return text

In [ ]:
def create_folders(base_path, dataset_name, n_levels=5):
    for level in range(n_levels):
        path = f"{base_path}/{dataset_name}/level{level}"
        os.makedirs(path, exist_ok=True)
    print(f" Folder {dataset_name} siap")


def save_folds(df, text_col, label_col, preprocess_fn, dataset_name, base_path, n_splits=5):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    X = df[text_col].values
    y = df[label_col].values

    for level in range(5):
        print(f"\n  Processing level {level}...")
        level_path = f"{base_path}/{dataset_name}/level{level}"

        for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):
            # Get data
            train_df = df.iloc[train_idx].copy()
            test_df = df.iloc[test_idx].copy()

            # Apply preprocessing
            train_df[text_col] = train_df[text_col].apply(lambda x: preprocess_fn(x, level))
            test_df[text_col] = test_df[text_col].apply(lambda x: preprocess_fn(x, level))

            # Save
            train_df.to_csv(f"{level_path}/fold{fold}_train.csv", index=False)
            test_df.to_csv(f"{level_path}/fold{fold}_test.csv", index=False)

        print(f"Level {level} Done! — {n_splits}")

In [ ]:
BASE_PATH = '/content/drive/MyDrive/riset_teks/preprocessed'

# Dataset 1: Reviews (ID)
print("=" * 50)
print("DATASET 1: Reviews (Bahasa Indonesia)")
print("=" * 50)
df1 = pd.read_csv('/content/drive/MyDrive/riset_teks/raw/dataset1-ulasan-5400-final.csv')
create_folders(BASE_PATH, 'dataset1')
save_folds(df1, 'text', 'class', preprocess_id, 'dataset1', BASE_PATH)

# Dataset 2: Hoax (ID)
print("\n" + "=" * 50)
print("DATASET 2: Hoax/Valid (Bahasa Indonesia)")
print("=" * 50)
df2 = pd.read_csv('/content/drive/MyDrive/riset_teks/raw/dataset2_hoax_valid_600-final.csv')
create_folders(BASE_PATH, 'dataset2')
save_folds(df2, 'text', 'class', preprocess_id, 'dataset2', BASE_PATH)

# Dataset 3: Medical Abstracts (EN)
print("\n" + "=" * 50)
print("DATASET 3: Medical Abstracts (English)")
print("=" * 50)
df3 = pd.read_csv('/content/drive/MyDrive/riset_teks/raw/dataset3-teknis-14438-final.csv')
create_folders(BASE_PATH, 'dataset3')
save_folds(df3, 'text', 'class', preprocess_en, 'dataset3', BASE_PATH)

print("\n Done! 150 files saved in Drive")

In [ ]:
# Check that the class distribution remains proportional in each fold.
def verify_fold(base_path, dataset_name, level=0, fold=1):
    train = pd.read_csv(f"{base_path}/{dataset_name}/level{level}/fold{fold}_train.csv")
    test = pd.read_csv(f"{base_path}/{dataset_name}/level{level}/fold{fold}_test.csv")
    print(f"\n{dataset_name} | Level {level} | Fold {fold}")
    print(f"Train: {train.shape} | Test: {test.shape}")
    print("Distribusi train:\n", train['class'].value_counts().sort_index())
    print("Distribusi test:\n", test['class'].value_counts().sort_index())

verify_fold(BASE_PATH, 'dataset1')
verify_fold(BASE_PATH, 'dataset2')
verify_fold(BASE_PATH, 'dataset3')